In [ ]:
# Fusion (Spatial + FCT(DST))
import os, cv2, numpy as np, random
import tensorflow as tf
from scipy.fft import dst
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

Mounted at /content/drive
Spatial: (10200, 128, 128, 1) FCT(DST): (10200, 128, 128, 1) labels: (10200,)
Epoch 1/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 201s 846ms/step - accuracy: 0.5755 - loss: 0.7367 - val_accuracy: 0.7868 - val_loss: 0.4564
Epoch 2/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 203s 851ms/step - accuracy: 0.7754 - loss: 0.4798 - val_accuracy: 0.8240 - val_loss: 0.3893
Epoch 3/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 194s 817ms/step - accuracy: 0.8137 - loss: 0.4078 - val_accuracy: 0.8152 - val_loss: 0.4112
Epoch 4/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 194s 828ms/step - accuracy: 0.8313 - loss: 0.3737 - val_accuracy: 0.8515 - val_loss: 0.3557
Epoch 5/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 209s 860ms/step - accuracy: 0.8964 - loss: 0.2526 - val_accuracy: 0.9706 - val_loss: 0.0724
Epoch 6/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 201s 856ms/step - accuracy: 0.9799 - loss: 0.0600 - val_accuracy: 0.9897 - val_loss: 0.0302
Epoch 7/20
234/234 ━━━━━━━━━━━━━━━━━━━━ 203s 868ms/step - accuracy: 0.9881 - loss: 0.0430 - val_accuracy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SEED=15
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
REAL_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/train"
FAKE_DIR = "/content/drive/My Drive/Research Project/StyleGAN2_7k/train"
IMG_SIZE=(128,128)

In [ ]:
def load_spatial_and_fct(real_dir, fake_dir, img_size=(128,128), limit=None):
    Xs, Xf, y = [], [], []
    def proc_sp(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        return img[..., None]
    def proc_fct(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        f = dst(dst(img, type=2, norm='ortho', axis=0), type=2, norm='ortho', axis=1)
        f = np.log(np.abs(f)+1e-8)
        #mn, mx = f.min(), f.max()
        #f = (f - mn) / (mx - mn + 1e-12)
        return f[..., None]
    reals = sorted(os.listdir(real_dir))
    fakes = sorted(os.listdir(fake_dir))
    if limit: reals, fakes = reals[:limit], fakes[:limit]
    for fn in reals:
        p = os.path.join(real_dir, fn)
        Xs.append(proc_sp(p)); Xf.append(proc_fct(p)); y.append(0)
    for fn in fakes:
        p = os.path.join(fake_dir, fn)
        Xs.append(proc_sp(p)); Xf.append(proc_fct(p)); y.append(1)
    return np.array(Xs), np.array(Xf), np.array(y)

In [ ]:
X_sp, X_fct, y = load_spatial_and_fct(REAL_DIR, FAKE_DIR)
print("Spatial:", X_sp.shape, "FCT(DST):", X_fct.shape, "labels:", y.shape)

In [ ]:
def build_branch(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(16,(4,4),activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64,activation='relu')(x)
    return models.Model(inp, x)

In [ ]:
def build_fusion(input_shape):
    a = build_branch(input_shape)
    b = build_branch(input_shape)
    comb = layers.Concatenate()([a.output, b.output])
    x = layers.Dense(64, activation='relu')(comb)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model([a.input, b.input], out)

In [ ]:
model = build_fusion((128,128,1))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
Xs_tr, Xs_te, Xf_tr, Xf_te, y_tr, y_te = train_test_split(X_sp, X_fct, y, test_size=0.2, random_state=SEED, stratify=y)

early_stop = EarlyStopping(monitor='val_loss', patience=9, restore_best_weights=True)
history = model.fit([Xs_tr, Xf_tr], y_tr, validation_data=([Xs_te, Xf_te], y_te), epochs=20, batch_size=35, callbacks=[early_stop], verbose=1)

print("Test eval:", model.evaluate([Xs_te, Xf_te], y_te, verbose=0))

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

REAL_TEST_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/test"
FAKE_TEST_DIR = "/content/drive/My Drive/Research Project/StyleGAN3_7k/test"

Xs_test, Xf_test, y_test = load_spatial_and_fct(REAL_TEST_DIR, FAKE_TEST_DIR, img_size=IMG_SIZE, limit=None)

print("StyleGAN3 Test Shapes:", Xs_test.shape, Xf_test.shape, y_test.shape)


test_loss, test_acc = model.evaluate([Xs_test, Xf_test], y_test, verbose=1)
print("StyleGAN3 Test Loss:", test_loss)
print("StyleGAN3 Test Accuracy:", test_acc)


y_pred_probs = model.predict([Xs_test, Xf_test])
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

print("\nClassification Report (StyleGAN3 Test Set):")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
StyleGAN3 Test Shapes: (4079, 128, 128, 1) (4079, 128, 128, 1) (4079,)
128/128 ━━━━━━━━━━━━━━━━━━━━ 24s 184ms/step - accuracy: 0.9790 - loss: 0.1043
StyleGAN3 Test Loss: 0.284282922744751
StyleGAN3 Test Accuracy: 0.9436136484146118
128/128 ━━━━━━━━━━━━━━━━━━━━ 22s 173ms/step

Classification Report (StyleGAN3 Test Set):
              precision    recall  f1-score   support

        Real       0.90      1.00      0.95      2079
        Fake       0.99      0.89      0.94      2000

    accuracy                           0.94      4079
   macro avg       0.95      0.94      0.94      4079
weighted avg       0.95      0.94      0.94      4079

